In [1]:
#!pip install keras_cv tensorflow_io

In [2]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
import os
import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Union, Tuple, List, Callable

# from google.colab import drive, files
# drive.mount('/content/drive')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)

In [3]:
# !cp -r "/content/drive/MyDrive/model" /content/
# !cp -r "/content/drive/MyDrive/Test" /content/
# !cp -r "/content/drive/MyDrive/images_test/images_spectograms" /content/

In [4]:
from src.model_trainer import ModelTrainer

model_name = "ResNet152V2" #"MobileNetV3Large" #"EfficientNetV2L" #"ResNet152V2" #"EfficientNetB7"

model_trainer = ModelTrainer(
    model_name=model_name,
    img_shape=(128, 256, 1),
    n_classes=667,
    dropout_rate=0.01,
    label_smoothing=0.3,
    fine_tune_layers=0
)

model = model_trainer.create_model()
model.summary()
model.load_weights(f"./models/weights_{model_name}.weights.h5")
tf.keras.backend.clear_session()

Model: "ResNet152V2_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 256, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ preprocess_input (Lambda)       │ (None, 128, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 2048)           │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ top_dropout (Dropout)           │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 667)            │     1,366,683 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,698,331 (227.73 MB)

 Trainable params: 59,554,587 (227.18 MB)

 Non-trainable params: 143,744 (561.50 KB)

/Users/camcortes/Documents/birds-sounds/.venv-birds-song/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 368 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
# cargar el label encoder del pickle
import pickle

with open(f'./models/label_encoder_{model_name}.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

In [6]:
from src.image_preprocessor import ImagePreprocessor

preprocessor = ImagePreprocessor(label_encoder=label_encoder)
data = preprocessor.load_data_from_directory("src/data/images_test/images_spectograms")
data.head()

/Users/camcortes/Documents/birds-sounds/.venv-birds-song/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,label,image_path
0,Acropternis orthonyx,src/data/images_test/images_spectograms/Acropternis orthonyx/121142_13.jpeg
1,Acropternis orthonyx,src/data/images_test/images_spectograms/Acropternis orthonyx/511703_5.jpeg
2,Acropternis orthonyx,src/data/images_test/images_spectograms/Acropternis orthonyx/461011_0.jpeg
3,Acropternis orthonyx,src/data/images_test/images_spectograms/Acropternis orthonyx/621780_9.jpeg
4,Acropternis orthonyx,src/data/images_test/images_spectograms/Acropternis orthonyx/428484_0.jpeg


In [7]:
import random
from tqdm import tqdm

def especie_batch(data):
    species = list(set(data["label"]))
    num_species = len(species)
    for i, specie in enumerate(tqdm(species, desc=f"Especies procesadas")):
        print(f"\nEspecie {i+1} de {(num_species)}, ({specie})")
        yield specie

def specie_ds(data, gen):
    try:
        specie = next(gen)
        df = data[data["label"] == specie]
        if df.empty:
            print(f"No more species")
            return None, None
    except StopIteration:
        return None, None
    return specie, preprocessor.create_validation_dataset(df)

In [8]:
import matplotlib.pyplot as plt
import numpy as np

def calcular_incertidumbre(dataset_espectrogramas, modelo):
    predicciones_mc = []
    confianzas_mc = []

    etiqueta_real_lista =[]
    etiqueta_inferencia_lista = []

    if dataset_espectrogramas:
        for imagenes_batch, etiquetas_batch in dataset_espectrogramas.take(1):
            num_imagenes = len(imagenes_batch)

            # MONTE CARLO DROPOUT: Múltiples pasadas con training=True
            for idx in range(num_imagenes):
                imagen = imagenes_batch[idx]
                # Añadir una dimensión de batch (necesario para la inferencia)
                imagen_batch = tf.expand_dims(imagen, axis=0)
                etiqueta_real = tf.argmax(etiquetas_batch[idx])

                etiqueta_real_lista.append(etiqueta_real.numpy())

                #for iteracion in range(5):
                logits = modelo(imagen_batch, training=True)
                probabilidades = tf.nn.softmax(logits, axis=-1).numpy()
                clase_predicha = np.argmax(probabilidades, axis=-1)[0]
                confianza = probabilidades[0, clase_predicha]

                predicciones_mc.append(clase_predicha)
                confianzas_mc.append(confianza)

                # Predicción convencional (sin dropout activo)
                logits_inferencia = modelo(imagen_batch, training=False)
                #logits_inferencia = modelo.predict(imagen_batch)
                probabilidades_inferencia = tf.nn.softmax(logits_inferencia, axis=-1).numpy()
                clase_predicha_inferencia = np.argmax(probabilidades_inferencia, axis=-1)[0]
                confianza_inferencia = probabilidades_inferencia[0, clase_predicha_inferencia]

                etiqueta_inferencia_lista.append(clase_predicha_inferencia)

                #print(f"\nClase real: {etiqueta_real}")
                #print(f"Clase predicha (modo inferencia): {clase_predicha_inferencia}")
                #print(f"Confianza (modo inferencia): {confianza_inferencia:.4f} ({confianza_inferencia*100:.2f}%)")

            # Imagen
            # plt.imshow(imagen.numpy().squeeze(), cmap='inferno')
            # plt.title(f"Predicha: {clase_predicha_inferencia} (Inferencia)")
            # plt.axis('off')
            # plt.show()

    return predicciones_mc, confianzas_mc, etiqueta_inferencia_lista, etiqueta_real_lista

In [9]:
dataset_generado = especie_batch(data)

resultados = {}
for i in range(data["label"].nunique()):
    especie, dataset_especie = specie_ds(data, dataset_generado)
    predicciones_mc, confianzas_mc, clase_predicha_inferencia, etiqueta_real = calcular_incertidumbre(dataset_especie, model)
    resultados[especie] = {
        "clase_real": etiqueta_real,
        "clase_predicha_inferencia": clase_predicha_inferencia,
        "predicciones_mc": predicciones_mc,
        "confianzas_mc": confianzas_mc,
    }

Especies procesadas:  87%|████████▋ | 583/667 [7:20:19<1:17:34, 55.41s/it]


Especie 584 de 667, (Ammodramus savannarum)


Especies procesadas:  88%|████████▊ | 584/667 [7:21:43<1:28:27, 63.95s/it]


Especie 585 de 667, (Myiarchus tuberculifer)


Especies procesadas:  88%|████████▊ | 585/667 [7:23:08<1:36:01, 70.26s/it]

...


Especie 665 de 667, (Euphonia chlorotica)


Especies procesadas: 100%|█████████▉| 665/667 [8:16:17<01:12, 36.37s/it]


Especie 666 de 667, (Sphenopsis melanotis)


Especies procesadas: 100%|█████████▉| 666/667 [8:16:38<00:31, 31.58s/it]


Especie 667 de 667, (Grallaria saturata)


In [15]:
# Alternativa más detallada si las longitudes de las listas varían
filas = []
for especie, datos in resultados.items():
    clase_real = datos["clase_real"]
    clase_predicha = datos["clase_predicha_inferencia"]
    predicciones = datos["predicciones_mc"]
    confianzas = datos["confianzas_mc"]

    max_len = max(len(clase_real), len(clase_predicha), len(predicciones), len(confianzas))

    for i in range(max_len):
        filas.append({
            "especie": especie,
            'registros': 1,
            "clase_real": clase_real[i] if i < len(clase_real) else None,
            "clase_predicha": clase_predicha[i] if i < len(clase_predicha) else None,
            "prediccion_mc": predicciones[i] if i < len(predicciones) else None,
            "confianza_mc": confianzas[i] if i < len(confianzas) else None,
            'true_positive': 1 if clase_real[i] == clase_predicha[i] else 0,
            'true_positivo_mc': 1 if clase_real[i] == predicciones[i] else 0,
        })

df_final = pd.DataFrame(filas)
print(df_final.shape)
#df_final

(32279, 8)


In [16]:
df_final.to_csv("src/data/incertidumbres_ResNet152V2.csv", index=False)

In [17]:
df = pd.read_csv("src/data/incertidumbres_ResNet152V2.csv")
df.head()

,especie,registros,clase_real,clase_predicha,prediccion_mc,confianza_mc,true_positive,true_positivo_mc
0,Catharus aurantiirostris,1,71,71,641,0.00,1,0
1,Catharus aurantiirostris,1,71,604,60,0.00,0,0
2,Catharus aurantiirostris,1,71,71,266,0.01,1,0
3,Catharus aurantiirostris,1,71,71,71,0.00,1,1
4,Catharus aurantiirostris,1,71,71,71,0.01,1,1


* Calcular la moda
* Calcular la distribución

In [18]:
df = df.groupby('especie').agg(
    {
        'registros': 'sum',
        'true_positive': 'mean',
        'true_positivo_mc': 'mean',
        'confianza_mc': 'max',

    }
).reset_index()

In [ ]:
def calcular_proporcion(row):
    real = set(row['clase_real'])
    predicha = set(row['clase_prediche_inferencia'])

    # Elementos comunes entre ambas listas
    elementos_comunes = len(real.intersection(predicha))

    # Total de elementos únicos en clase_real
    total_elementos_real = len(real)

    # Calcula la proporción (evitando división por cero)
    if total_elementos_real > 0:
        return elementos_comunes / total_elementos_real
    else:
        return 0

df['proporcion_coincidencia'] = df.apply(calcular_proporcion, axis=1)
df['porcentaje_coincidencia'] = df['proporcion_coincidencia'] * 100

In [19]:
df.to_csv("/content/drive/MyDrive/Incertidumbres/incertidumbres_ResNet152V2.csv")
#guardar excel
df.to_excel("/content/drive/MyDrive/Incertidumbres/incertidumbres_ResNet152V2.xlsx")

In [ ]:
all_probs = np.concatenate(probabilities, axis=0)
mean_prob = np.mean(all_probs, axis=0)
# Entropía de la distribución promedio
predictive_entropy = -np.sum(mean_prob * np.log(mean_prob + 1e-9))
print("Entropía de predicción:", predictive_entropy)

In [ ]:
np.exp(0.00974931)